In [442]:
import plotly.express as px
import plotly.graph_objects as go
import numpy as np
import pandas as pd
from scipy.interpolate import griddata

In [443]:
# df = pd.read_csv(r'C:\\Users\\canel\\OneDrive\Desktop\\ToyData.csv', skiprows=3)
df = pd.read_csv('ToyData2.csv')
print(df.shape)
df.head(20)

(40000, 6)


,id,instance_no,reg_type,k_repetition,reg_value,loss
0,0,0.0,L2,0.0,0.00100,0.960612
1,1,0.0,L2,1.0,0.00100,0.801498
2,2,0.0,L2,2.0,0.00100,1.074984
3,3,0.0,L2,3.0,0.00100,0.919001
4,4,0.0,L2,4.0,0.00100,0.957142
5,5,0.0,L2,5.0,0.00100,0.909571
6,6,0.0,L2,6.0,0.00100,0.996477
7,7,0.0,L2,7.0,0.00100,0.966306
8,8,0.0,L2,8.0,0.00100,1.031313
9,9,0.0,L2,9.0,0.00100,0.954000


In [444]:
unique_b_no = df['instance_no'].unique()
unique_reg_types = df['reg_type'].unique()
unique_reg_values = df.groupby('reg_type')['reg_value'].unique()
# unique_reg1, unique_reg2 = unique_reg_values[0], unique_reg_values[1]
print(unique_b_no)
print(unique_reg_types)
print(unique_reg_values)
print(df.columns)

[ 0.  1.  2.  3.  4.  5.  6.  7.  8.  9. 10. 11. 12. 13. 14. 15. 16. 17.
 18. 19.]
['L2' 'dropout']
reg_type
L2         [0.001, 0.0030204081632653, 0.0050408163265306...
dropout    [0.1, 0.1081632653061224, 0.1163265306122449, ...
Name: reg_value, dtype: object
Index(['id', 'instance_no', 'reg_type', 'k_repetition', 'reg_value', 'loss'], dtype='object')


In [445]:
summary_df = pd.DataFrame(columns=['instance_no','reg_type', 'reg_value', 'mean_loss', 'stdv_loss'])

for b_value in unique_b_no:
    # print(b_value)

    for reg_type in unique_reg_types:
        # print(reg_type)

        for reg_values in unique_reg_values:
            # print(reg_values)

            for value in reg_values:
                # print(value)
                selected_rows = df[(df['instance_no'] == b_value) & (df['reg_type'] == reg_type) & (df['reg_value'] == value)]
                # print(selected_rows.shape)
                # input()
                mean_loss = selected_rows['loss'].mean()
                stdv_loss = selected_rows['loss'].std()
                # print(f'{b_value}, {reg_type}, {reg_value}, {mean_loss}')
                new_row = {'instance_no': b_value, 'reg_type': reg_type, 'reg_value': value, 'mean_loss': mean_loss, 'stdv_loss': stdv_loss}
                summary_df = pd.concat([summary_df, pd.DataFrame([new_row])], ignore_index=True)


summary_df = summary_df.dropna()
print(f'Summary shape: {summary_df.shape}')
print(summary_df.head(60))
# print(summary_df.tail(10))

Summary shape: (408, 5)
     instance_no reg_type  reg_value  mean_loss  stdv_loss
0            0.0       L2   0.001000   0.992337   0.093475
1            0.0       L2   0.003020   1.002284   0.113245
10           0.0       L2   0.021204   0.999835   0.099432
11           0.0       L2   0.023224   0.994664   0.107481
20           0.0       L2   0.041408   0.992220   0.100211
21           0.0       L2   0.043429   0.996341   0.104607
30           0.0       L2   0.061612   1.012744   0.100973
31           0.0       L2   0.063633   1.017362   0.108450
40           0.0       L2   0.081816   1.022780   0.103737
41           0.0       L2   0.083837   1.002543   0.097748
149          0.0  dropout   0.100000   0.997143   0.107978
150          0.0  dropout   0.100000   0.997143   0.107978
151          0.0  dropout   0.108163   1.008665   0.101732
160          0.0  dropout   0.181633   1.030721   0.096994
161          0.0  dropout   0.189796   1.020632   0.100122
170          0.0  dropout   0.26

In [446]:
summary_df.columns

Index(['instance_no', 'reg_type', 'reg_value', 'mean_loss', 'stdv_loss'], dtype='object')

In [447]:
# summary_unique_reg_types = df['reg_type'].unique()
# # print(summary_unique_reg_types)

# def split_dataframe(df, feature, value):
#     df_split = df([df[feature] == value])
#     return df_split    

In [448]:
# Assuming you have a dataframe called summary_df
# with columns ['instance_no', 'reg_type', 'reg_value', 'mean_loss', 'stdv_loss']

# Create a dictionary of dataframes to store the split dataframes
split_dataframes = {}

# Group the dataframe by 'reg_type'
grouped = summary_df.groupby('reg_type')

# Iterate over each group
for reg_type, group_df in grouped:
    # Store the dataframe with the current 'reg_type' in the dictionary
    split_dataframes[reg_type] = group_df

# Now split_dataframes contains multiple dataframes, each with a unique value of 'reg_type'



In [449]:

# len(split_dataframes)

In [450]:
# split_dataframes['L2'].shape

In [451]:
def plot_3d(df_dict, output_variable):
    
    for key, df in df_dict.items():
        # print(f'Key {key}')
        # print(f'Value: {df}') 

        # Define grid for surface
        b_no_range = np.linspace(df['instance_no'].min(), df['instance_no'].max(), 100)
        reg_value_range = np.linspace(df['reg_value'].min(), df['reg_value'].max(), 100)
        b_no_grid, reg_value_grid = np.meshgrid(b_no_range, reg_value_range)

        # Interpolate mean_loss values for the surface
        
        # print(f'Len instance: {df['instance_no'].shape}')
        # print(f'Len reg val: {df['reg_value'].shape}')
        
        mean_loss_surface = griddata((df['instance_no'], df['reg_value']), 
                                    df[output_variable], 
                                    (b_no_grid, reg_value_grid), 
                                    method='cubic')

        # Create the 3D surface plot
        fig = go.Figure()

        # Add scatter data points
        fig.add_trace(go.Scatter3d(
            x=df['instance_no'],
            y=df['reg_value'],
            z=df[output_variable],
            mode='markers',
            marker=dict(
                size=3,
                color='blue',
                opacity=0.5
            ),
            name='Data Points'
        ))

        # Add surface plot
        fig.add_trace(go.Surface(
            x=b_no_range,
            y=reg_value_range,
            z=mean_loss_surface,
            colorscale='Jet',
            opacity=0.6
        ))

        # Update layout for better visibility
        fig.update_layout(scene=dict(
                            xaxis_title='instance_no',
                            yaxis_title='reg_value',
                            zaxis_title= output_variable),
                            width=800,  # adjust width
                            height=600, # adjust height
                            title=f'Mean Loss for different values of {key} regularization')

        # Show the plot
        fig.show()


In [452]:
plot_3d(split_dataframes, 'mean_loss')

In [453]:
# plot_3d(split_dataframes, 'stdv_loss')

In [454]:
# Group by 'reg_type' and 'reg_value'
grouped = summary_df.groupby('reg_type')#, 'reg_value'])

# Create a dictionary to store DataFrames for each combination
dfs = {}

# Iterate over the groups and create a DataFrame for each combination
for name, group in grouped:
    reg_type = name
    df_name = f"{reg_type}"  # Name for the DataFrame
    dfs[df_name] = group.copy()  # Copy the group DataFrame and store it


In [455]:
# for df_name, df in dfs.items():
#     print(f"DataFrame: {df_name}")
#     print(df.shape)
#     print(df)
#     print("\n")

In [456]:
def plot_2d(df_dict, output_variable):
    for key, df in df_dict.items():
        # Group by 'reg_type'
        grouped_by_reg_type = summary_df.groupby('reg_type')

        # Iterate over each group of reg_type
        for reg_type, reg_type_group in grouped_by_reg_type:
            # Group by 'reg_value' within each reg_type group
            grouped_by_reg_value = reg_type_group.groupby('reg_value')
            
            # Create a figure for the current reg_type
            fig = go.Figure()
            
            # Iterate over each group of reg_value within the current reg_type group
            for reg_value, reg_value_group in grouped_by_reg_value:
                fig.add_trace(go.Scatter(x=reg_value_group['instance_no'], y=reg_value_group['mean_loss'], 
                                        mode='lines+markers', name=reg_value,
                                        marker=dict(size=7)))
            
            fig.update_layout(legend=dict(title="Reg. Value"))
            fig.update_layout(title=f'Iterations vs. mean loss for {reg_type}',
                            xaxis_title='iteration',
                            yaxis_title='mean_loss',
                            width=600,  # Set width of the plot
                            height=400)  # Set height of the plot
            
            # Show the plot for the current reg_type
            fig.show()


In [457]:
plot_2d(split_dataframes, 'stdv_loss')

In [458]:
# # Group by 'reg_type'
# grouped_by_reg_type = summary_df.groupby('reg_type')

# # Iterate over each group of reg_type
# for reg_type, reg_type_group in grouped_by_reg_type:
#     # Group by 'reg_value' within each reg_type group
#     grouped_by_reg_value = reg_type_group.groupby('reg_value')
    
#     # Create a figure for the current reg_type
#     fig = go.Figure()
    
#     # Iterate over each group of reg_value within the current reg_type group
#     for reg_value, reg_value_group in grouped_by_reg_value:
#         fig.add_trace(go.Scatter(x=reg_value_group['instance_no'], y=reg_value_group['mean_loss'], 
#                                  mode='lines+markers', name=reg_value,
#                                  marker=dict(size=7)))
    
#     fig.update_layout(legend=dict(title="Reg. Value"))
#     fig.update_layout(title=f'Iterations vs. mean loss for {reg_type}',
#                       xaxis_title='iteration',
#                       yaxis_title='mean_loss',
#                       width=600,  # Set width of the plot
#                       height=400)  # Set height of the plot
    
#     # Show the plot for the current reg_type
#     fig.show()


In [459]:
# Group by 'reg_type'
# grouped_by_reg_type = value.groupby('reg_type')

# Iterate over each group of reg_type
for reg_type, reg_type_group in grouped_by_reg_type:
    # Group by 'reg_value' within each reg_type group
    grouped_by_reg_value = reg_type_group.groupby('reg_value')
    
    # Create a figure for the current reg_type
    fig = go.Figure()
    
    # Iterate over each group of reg_value within the current reg_type group
    for reg_value, reg_value_group in grouped_by_reg_value:
        fig.add_trace(go.Scatter(x=reg_value_group['instance_no'], y=reg_value_group['stdv_loss'], 
                                 mode='lines+markers', name=reg_value,
                                 marker=dict(size=7)))
    
    fig.update_layout(legend=dict(title="Reg. Value"))
    fig.update_layout(title=f'Iterations vs. stdv loss for {reg_type}',
                      xaxis_title='iteration',
                      yaxis_title='stdv_loss',
                      width=600,  # Set width of the plot
                      height=400)  # Set height of the plot
    
    # Show the plot for the current reg_type
    fig.show()
